In [70]:
##Loading Data
from langchain_community.document_loaders import WebBaseLoader
web_loader = WebBaseLoader(web_path="https://docs.langchain.com/langsmith/evaluation-quickstart")
docs = web_loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-quickstart', 'title': 'Evaluation quickstart - Docs by LangChain', 'language': 'en'}, page_content='Evaluation quickstart - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageTestSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationEvaluation quickstartGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewQuickstartConceptsEvaluation approachesOn this pagePrerequisitesEvaluation quickstartCopy pageCopy pageCopy pageCopy pageEvaluations are a quantitative way to measure the performance of LLM applications. LLMs can behave unpredictably, even small changes 

In [71]:
## Splitting Data

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000 , chunk_overlap = 50)
text_chunks = text_splitter.split_documents(docs)
text_chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-quickstart', 'title': 'Evaluation quickstart - Docs by LangChain', 'language': 'en'}, page_content="Evaluation quickstart - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageTestSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationEvaluation quickstartGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewQuickstartConceptsEvaluation approachesOn this pagePrerequisitesEvaluation quickstartCopy pageCopy pageCopy pageCopy pageEvaluations are a quantitative way to measure the performance of LLM applications. LLMs can behave unpredictably, even small changes t

In [72]:
## Loading our llm basically langchain llm wrapper for gemini

import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['GEMINI_API_KEY'] = os.getenv("GEMINI_API_KEY")
os.environ['LANGSMITH_API_KEY'] = os.getenv("LANGSMITH_API_KEY")
os.environ['LANGSMITH_ENDPOINT'] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")


In [73]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
db = Chroma.from_documents(text_chunks, embeddings)
db

In [74]:
##Initialize 

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.5'}} profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') model='gemini-2.5-flash' client=<google.genai.client.Client object at 0x000001E98544B3B0> default_metadata=() model_kwargs={}


In [75]:
query = "Evaluations are a quantitative way to measure the performance of LLM applications."
result = db.similarity_search(query)
result[0].page_content

'quickstartGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewQuickstartConceptsEvaluation approachesOn this pagePrerequisitesEvaluation quickstartCopy pageCopy pageCopy pageCopy pageEvaluations are a quantitative way to measure the performance of LLM applications. LLMs can behave unpredictably, even small changes to prompts, models, or inputs can significantly affect results. Evaluations provide a structured way to identify failures, compare'

In [76]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.5'}} profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') model='gemini-2.5-flash' client=<google.genai.client.Client object at 0x000001E985453140> default_metadata=() model_kwargs={}


In [77]:
## Retrival Chain , Document Chain 

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the following based on the context below, and if the answer is not contained within the text below, say "I don't know"

<context>
{context}
</context>

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following based on the context below, and if the answer is not contained within the text below, say "I don\'t know"\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.5'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'aud

In [78]:
question = "Evaluations are a quantitative way to measure the performance of LLM applications."

document_chain.invoke({"input": question,
                       "context": db.similarity_search(question)})

'Yes'

In [79]:
## Input -> Retriver -> db
db

In [80]:
retriever = db.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retriever_chain = create_retrieval_chain(retriever, document_chain)
retriever_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001E982691310>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following based on the context below, and if the answer is not contained within the text below, say "I don\'t kno

In [85]:
### Get the response from the llm
res = retriever_chain.invoke({
    "input" : "Evaluations are a quantitative way to measure the performance of LLM applications."
})
res["answer"]
res
res['context']

[Document(id='6274624f-dc47-402b-adae-153f7a50f217', metadata={'title': 'Evaluation quickstart - Docs by LangChain', 'source': 'https://docs.langchain.com/langsmith/evaluation-quickstart', 'language': 'en'}, page_content='quickstartGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewQuickstartConceptsEvaluation approachesOn this pagePrerequisitesEvaluation quickstartCopy pageCopy pageCopy pageCopy pageEvaluations are a quantitative way to measure the performance of LLM applications. LLMs can behave unpredictably, even small changes to prompts, models, or inputs can significantly affect results. Evaluations provide a structured way to identify failures, compare'),
 Document(id='aa9ffc92-5d7d-4ad5-9c89-d41c3ad0b92d', metadata={'title': 'Evaluation quickstart - Docs by LangChain', 'language': 'en', 'source': 'https://docs.langchain.com/langsmith/evaluation-quickstart'}, page_content='quickstartGet startedDatasets & ExperimentsEvaluatorsA